# Email Spam Dataset — Preprocessing Pipeline

This notebook prepares Datasets A, B, and C for the separate within-corpus and cross-corpus evaluation notebooks. It is intentionally limited to **data inspection, raw-text metadata extraction, text cleaning, validation, and saving the processed data**. Model training and TF-IDF fitting belong in the evaluation notebooks so that vectorizers are fitted only on the appropriate training partition/source corpus.

## Preprocessing order

1. Load the raw dataset and standardize the text/label column names.
2. Inspect missing values and exact duplicate messages.
3. Handle missing values according to the explicitly selected policy below.
4. For Dataset C, remove exact duplicate raw messages after newline normalization. For Datasets A and B, report exact duplicates but retain them, matching the current manuscript description.
5. Extract metadata from the raw email text: URL occurrences, email-address occurrences, exclamation marks, dollar signs, hash symbols, and numerical digits.
6. Normalize the text: lowercase → remove characters outside `a-z`/whitespace → split into tokens → remove English stopwords → apply Porter stemming.
7. Validate the label column and save the processed data.

**Important:** this notebook does not apply `StandardScaler`. Metadata values remain count-based numerical features and can be passed through unchanged in the downstream pipelines. It also does not create a global TF-IDF matrix.


## 1. Imports and NLTK resources

The imports are limited to preprocessing and data inspection. The stopword list uses NLTK's English stopwords and the stemmer is NLTK's Porter stemmer, matching the existing pipeline.


In [ ]:
from pathlib import Path
import re
import json
import warnings

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

try:
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    # Download once if the NLTK corpus is not already installed in this environment.
    nltk.download("stopwords", quiet=True)
    try:
        STOP_WORDS = set(stopwords.words("english"))
    except LookupError as exc:
        raise RuntimeError(
            "NLTK English stopwords are unavailable. Run nltk.download('stopwords') "
            "in a network-enabled session, then rerun this cell."
        ) from exc

STEMMER = PorterStemmer()
print(f"Loaded {len(STOP_WORDS)} English stopwords.")


## 2. Settings — update the raw input paths

Set each path to the original/raw dataset, not a previously cleaned sample, unless you deliberately want to reprocess that sample. Supported formats are CSV, pickle (`.pkl`/`.pickle`), and Parquet. The column names are configurable; `text_column=None` tries to identify `text` or `text_combined` automatically and raises an error if the choice is ambiguous.

- Datasets A and B: exact duplicates are **reported and retained**.
- Dataset C: exact duplicates are **removed**, based on the raw message text after newline normalization.
- The default missing-value policy is `"error"`: if missing text/labels are present, the notebook stops rather than guessing what the historical imputation was. After inspecting the audit, change the policy only if appropriate.


In [ ]:
DATASET_CONFIG = {
    "A": {
        "path": "",                 # TODO: enter the path to raw Dataset A
        "text_column": None,         # e.g. "text_combined" or "text"
        "label_column": "label",
        "remove_exact_duplicates": False,
    },
    "B": {
        "path": "",                 # TODO: enter the path to raw Dataset B
        "text_column": None,         # e.g. "text_combined" or "text"
        "label_column": "label",
        "remove_exact_duplicates": False,
    },
    "C": {
        "path": "",                 # TODO: enter the path to raw Dataset C (e.g. dataset3.csv)
        "text_column": "text",
        "label_column": "label",
        "remove_exact_duplicates": True,
    },
}

# Accepted options: "error", "fill_empty_string", "drop_rows".
# Keep "error" until the missing-value audit has been reviewed.
MISSING_TEXT_POLICY = "error"
MISSING_LABEL_POLICY = "error"

# Output directory. In Kaggle, files saved here are written to the working directory.
OUTPUT_DIR = Path("preprocessed_output")

METADATA_FEATURES = [
    "num_exclamations",
    "num_dollars",
    "num_urls",
    "num_emails",
    "num_hashes",
    "num_digits",
]

TEXT_COLUMN_CANDIDATES = ["text", "text_combined"]
LABEL_COLUMN_CANDIDATES = ["label"]

print("Configured datasets:")
for dataset_name, settings in DATASET_CONFIG.items():
    print(f"  Dataset {dataset_name}: path={settings['path']!r}")


## 4. Helper functions: loading, column checks, and raw-text preparation

Newlines are changed to spaces before duplicate checking, consistent with the existing Dataset C notebook. This step only standardizes line breaks; it does not lowercase, strip punctuation, or otherwise clean the text before metadata extraction.


In [ ]:
def load_dataframe(path_value):
    """Load a DataFrame from CSV, pickle, or Parquet."""
    if not path_value or not str(path_value).strip():
        raise ValueError("A dataset path is blank. Set it in DATASET_CONFIG before loading.")

    path = Path(path_value)
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset file not found: {path}. Update the corresponding path in DATASET_CONFIG."
        )

    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file extension {suffix!r} for {path}.")


def resolve_column(df, configured_name, candidates, role, dataset_name):
    """Resolve a configured column or safely infer it when there is one match."""
    if configured_name is not None:
        if configured_name not in df.columns:
            raise KeyError(
                f"Dataset {dataset_name}: configured {role} column {configured_name!r} was not found. "
                f"Available columns: {list(df.columns)}"
            )
        return configured_name

    found = [column for column in candidates if column in df.columns]
    if len(found) == 1:
        return found[0]
    if not found:
        raise KeyError(
            f"Dataset {dataset_name}: could not infer the {role} column. "
            f"Set it explicitly in DATASET_CONFIG. Available columns: {list(df.columns)}"
        )
    raise ValueError(
        f"Dataset {dataset_name}: multiple possible {role} columns were found: {found}. "
        f"Set the correct column explicitly in DATASET_CONFIG."
    )


def normalize_newlines(value):
    """Replace newline variants with spaces without cleaning other characters."""
    if not isinstance(value, str):
        return value
    return value.replace("\r\n", "\n").replace("\r", "\n").replace("\n", " ")


def load_and_standardize_dataset(dataset_name, settings):
    raw_df = load_dataframe(settings["path"])
    text_col = resolve_column(
        raw_df,
        settings.get("text_column"),
        TEXT_COLUMN_CANDIDATES,
        "text",
        dataset_name,
    )
    label_col = resolve_column(
        raw_df,
        settings.get("label_column"),
        LABEL_COLUMN_CANDIDATES,
        "label",
        dataset_name,
    )

    # Keep only the fields needed by the documented preprocessing pipeline.
    standardized = raw_df[[text_col, label_col]].copy()
    standardized.columns = ["raw_text", "label"]
    standardized["raw_text"] = standardized["raw_text"].map(normalize_newlines)

    info = {
        "dataset": dataset_name,
        "source_path": str(settings["path"]),
        "raw_rows": int(len(raw_df)),
        "original_text_column": text_col,
        "original_label_column": label_col,
    }
    return standardized, info


## 5. Load datasets and audit missing values / duplicates

Review this output before continuing. In particular, verify the number of duplicate messages in each dataset and the missing-value counts. The code does **not** silently impute missing data.


In [ ]:
raw_frames = {}
load_info = {}

for dataset_name, settings in DATASET_CONFIG.items():
    raw_frames[dataset_name], load_info[dataset_name] = load_and_standardize_dataset(
        dataset_name, settings
    )

    frame = raw_frames[dataset_name]
    text_missing = frame["raw_text"].isna() | frame["raw_text"].astype("string").str.strip().eq("")
    label_missing = frame["label"].isna()
    duplicate_rows = int(frame["raw_text"].duplicated(keep="first").sum())

    load_info[dataset_name].update({
        "missing_text_or_blank": int(text_missing.sum()),
        "missing_label": int(label_missing.sum()),
        "exact_duplicate_text_rows": duplicate_rows,
    })

missing_duplicate_audit = pd.DataFrame(load_info.values()).set_index("dataset")
display(missing_duplicate_audit)


### Optional A/B exact-overlap audit

This computes the number of shared unique raw messages between A and B after newline normalization. It is an audit only; it does not remove any records. It is useful for checking the exact-overlap analysis described in the manuscript.


In [ ]:
if "A" in raw_frames and "B" in raw_frames:
    a_text = set(raw_frames["A"]["raw_text"].dropna().astype(str))
    b_text = set(raw_frames["B"]["raw_text"].dropna().astype(str))
    shared_messages = a_text.intersection(b_text)
    print(f"Unique raw messages shared by A and B: {len(shared_messages):,}")
    print(f"Unique messages in A: {len(a_text):,}")
    print(f"Unique messages in B: {len(b_text):,}")
else:
    print("Load both Dataset A and Dataset B to run the overlap audit.")


## 6. Metadata extraction and text cleaning functions

Metadata is extracted from the raw email message **before** the text normalization below. The metadata values are counts, not binary indicators. These columns are not scaled in this notebook; the downstream model pipeline may pass them through unchanged.

The text-cleaning expression below preserves the current method: `re.sub(r'[^a-z\s]', '', lowercased_text)`, English stopword removal, then Porter stemming. As a result, pre-existing alphabetic placeholders such as `escapenumb` and `escapelong` remain as text tokens.


In [ ]:
def extract_special_features(raw_text):
    """Extract count-based metadata directly from the raw message text."""
    text = str(raw_text)
    return {
        "num_exclamations": text.count("!"),
        "num_dollars": text.count("$"),
        "num_urls": len(re.findall(r"http[s]?://\S+", text)),
        "num_emails": len(re.findall(r"\S+@\S+", text)),
        "num_hashes": text.count("#"),
        "num_digits": sum(character.isdigit() for character in text),
    }


def clean_email_text(raw_text):
    """Apply lowercase, character cleaning, stopword removal, and Porter stemming."""
    text = str(raw_text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    tokens = [STEMMER.stem(word) for word in tokens if word not in STOP_WORDS]
    return " ".join(tokens)

# Quick sanity check of the helpers; these examples do not modify any dataset.
_example = "URGENT!!! Contact help@example.com or visit https://example.com/pay $50 #offer"
print("Metadata example:", extract_special_features(_example))
print("Cleaned-text example:", clean_email_text(_example))


## 7. Apply missing-value policy, duplicate policy, and preprocessing

Policies are explicit and applied separately to text and labels:

- `error`: stop if missing values are detected, so they can be investigated rather than handled silently.
- `fill_empty_string` (text only): replace missing/blank message text with an empty string.
- `drop_rows`: remove records missing the relevant text or label.

For labels, do not invent replacement values. After missing-value handling, labels must be encoded as numeric `0` and `1` (legitimate/ham = `0`, spam = `1`).

Dataset A and B exact duplicates are counted and retained. Dataset C duplicates are removed by exact `raw_text` matching after newline normalization, as in the existing LLM notebook.


In [ ]:
def apply_missing_value_policy(frame, dataset_name):
    frame = frame.copy()
    text_missing = frame["raw_text"].isna() | frame["raw_text"].astype("string").str.strip().eq("")
    label_missing = frame["label"].isna()

    if MISSING_TEXT_POLICY not in {"error", "fill_empty_string", "drop_rows"}:
        raise ValueError("MISSING_TEXT_POLICY must be 'error', 'fill_empty_string', or 'drop_rows'.")
    if MISSING_LABEL_POLICY not in {"error", "drop_rows"}:
        raise ValueError("MISSING_LABEL_POLICY must be 'error' or 'drop_rows'.")

    if text_missing.any():
        if MISSING_TEXT_POLICY == "error":
            raise ValueError(
                f"Dataset {dataset_name} has {int(text_missing.sum())} missing/blank text values. "
                "Review the audit and choose the text policy in the settings cell before continuing."
            )
        if MISSING_TEXT_POLICY == "fill_empty_string":
            frame.loc[text_missing, "raw_text"] = ""
        elif MISSING_TEXT_POLICY == "drop_rows":
            frame = frame.loc[~text_missing].copy()

    label_missing = frame["label"].isna()
    if label_missing.any():
        if MISSING_LABEL_POLICY == "error":
            raise ValueError(
                f"Dataset {dataset_name} has {int(label_missing.sum())} missing labels. "
                "Inspect the records; do not invent label values. Choose the label policy only after review."
            )
        frame = frame.loc[~label_missing].copy()

    # Convert non-null values to strings after the missing-value decision.
    frame["raw_text"] = frame["raw_text"].astype(str)
    frame["label"] = pd.to_numeric(frame["label"], errors="raise")

    unexpected_labels = sorted(set(frame["label"].unique()) - {0, 1})
    if unexpected_labels:
        raise ValueError(
            f"Dataset {dataset_name} has labels outside the expected 0/1 encoding: {unexpected_labels}. "
            "Map the original labels explicitly before continuing."
        )
    frame["label"] = frame["label"].astype(int)
    return frame


def preprocess_dataset(dataset_name, source_frame, settings):
    frame = apply_missing_value_policy(source_frame, dataset_name)
    rows_after_missing_handling = len(frame)

    duplicate_rows_found = int(frame["raw_text"].duplicated(keep="first").sum())
    duplicates_removed = 0
    if settings.get("remove_exact_duplicates", False):
        frame = frame.drop_duplicates(subset=["raw_text"], keep="first").copy()
        duplicates_removed = rows_after_missing_handling - len(frame)
    else:
        # Dataset A/B duplicate messages are retained, consistent with the manuscript.
        if duplicate_rows_found:
            warnings.warn(
                f"Dataset {dataset_name}: retaining {duplicate_rows_found:,} exact duplicate text rows, "
                "as specified by the current methodology.",
                RuntimeWarning,
            )

    raw_texts = frame["raw_text"].tolist()
    metadata_rows = [extract_special_features(text) for text in raw_texts]
    cleaned_texts = [clean_email_text(text) for text in raw_texts]
    metadata_df = pd.DataFrame(metadata_rows, columns=METADATA_FEATURES, index=frame.index)

    processed = pd.concat(
        [
            pd.Series(cleaned_texts, index=frame.index, name="text"),
            metadata_df,
            frame["label"].rename("label"),
        ],
        axis=1,
    ).reset_index(drop=True)

    if processed["text"].isna().any():
        raise AssertionError(f"Unexpected missing cleaned text found in Dataset {dataset_name}.")
    if processed["label"].isna().any():
        raise AssertionError(f"Unexpected missing labels found in Dataset {dataset_name}.")
    if not set(processed["label"].unique()).issubset({0, 1}):
        raise AssertionError(f"Dataset {dataset_name} labels are not binary 0/1.")

    summary = {
        "dataset": dataset_name,
        "rows_before_missing_handling": int(len(source_frame)),
        "rows_after_missing_handling": int(rows_after_missing_handling),
        "exact_duplicate_text_rows_found": int(duplicate_rows_found),
        "duplicates_removed": int(duplicates_removed),
        "processed_rows": int(len(processed)),
        "ham_label_0": int((processed["label"] == 0).sum()),
        "spam_label_1": int((processed["label"] == 1).sum()),
        "unique_cleaned_texts": int(processed["text"].nunique(dropna=False)),
    }
    return processed, summary


## 8. Process the configured datasets

This cell creates a processed DataFrame for each source. If a dataset has missing values and the policy is still `"error"`, execution stops with a message describing what must be reviewed.


In [ ]:
processed_frames = {}
processing_summaries = []

for dataset_name, settings in DATASET_CONFIG.items():
    print(f"\nPreprocessing Dataset {dataset_name}...")
    processed_frames[dataset_name], summary = preprocess_dataset(
        dataset_name,
        raw_frames[dataset_name],
        settings,
    )
    processing_summaries.append(summary)
    print(f"Processed shape: {processed_frames[dataset_name].shape}")
    print(processed_frames[dataset_name]["label"].value_counts().sort_index().rename(index={0: "ham (0)", 1: "spam (1)"}))

processing_summary_df = pd.DataFrame(processing_summaries).set_index("dataset")
display(processing_summary_df)


## 9. Validate the processed datasets

The expected final schema is `text`, the six count-based metadata fields, and `label`. The text column is the normalized/stemmed representation. There is no raw-text column in the output, so downstream code that uses `df.drop(columns=["label"])` will not accidentally include raw text or row identifiers as model features.


In [ ]:
expected_columns = ["text", *METADATA_FEATURES, "label"]

for dataset_name, processed in processed_frames.items():
    if list(processed.columns) != expected_columns:
        raise AssertionError(
            f"Dataset {dataset_name} has unexpected columns: {list(processed.columns)}"
        )
    if not all(pd.api.types.is_numeric_dtype(processed[col]) for col in METADATA_FEATURES):
        raise AssertionError(f"Dataset {dataset_name} metadata columns must be numeric.")

    print(f"\nDataset {dataset_name}: {len(processed):,} rows")
    print("Columns:", list(processed.columns))
    print("Missing values by column:")
    display(processed.isna().sum().to_frame("missing_values"))
    display(processed.head(3))


## 10. Save processed data and audit reports

Processed datasets are saved as both pickle (recommended for subsequent Python notebooks) and CSV (easy to inspect). The summary and pre-processing audit are also exported. The files are saved locally under `preprocessed_output/`; in Kaggle, download them from the notebook's Output/Files panel or save the directory as notebook output.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for dataset_name, processed in processed_frames.items():
    pickle_path = OUTPUT_DIR / f"Dataset_{dataset_name}_preprocessed.pkl"
    csv_path = OUTPUT_DIR / f"Dataset_{dataset_name}_preprocessed.csv"
    processed.to_pickle(pickle_path)
    processed.to_csv(csv_path, index=False)
    print(f"Dataset {dataset_name} saved to:\n  {pickle_path}\n  {csv_path}")

processing_summary_path = OUTPUT_DIR / "preprocessing_summary.csv"
processing_summary_df.to_csv(processing_summary_path)
missing_audit_path = OUTPUT_DIR / "input_missing_duplicate_audit.csv"
missing_duplicate_audit.to_csv(missing_audit_path)

print(f"\nSummary saved to: {processing_summary_path}")
print(f"Input audit saved to: {missing_audit_path}")
